In [61]:
import pandas as pd 

df = pd.read_csv('combined_output.csv', low_memory=False)
df.head()

,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,...,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
0,2020-02-19T02:05:42Z,"Android OS 9 API 28 (samsung, SM-A505F)",132050,IN,47.15.173.165,Changes,Justin Bieber,Changes,spotify:track:521Os1SMlxmsoevsqyjNmJ,NaN,...,NaN,NaN,NaN,clickrow,endplay,False,False,False,NaN,False
1,2020-02-19T02:10:06Z,"Android OS 9 API 28 (samsung, SM-A505F)",255861,IN,47.15.173.165,Second Emotion (feat. Travis Scott),Justin Bieber,Changes,spotify:track:1PnAx788sAvcI3ZxTlEyX8,NaN,...,NaN,NaN,NaN,clickrow,trackdone,False,False,False,NaN,False
2,2020-02-19T02:10:07Z,"Android OS 9 API 28 (samsung, SM-A505F)",0,IN,47.15.173.165,Second Emotion (feat. Travis Scott),Justin Bieber,Changes,spotify:track:1PnAx788sAvcI3ZxTlEyX8,NaN,...,NaN,NaN,NaN,trackdone,endplay,False,False,False,NaN,False
3,2020-02-19T02:10:26Z,"Android OS 9 API 28 (samsung, SM-A505F)",2820,IN,47.15.173.165,In These Arms,JP Cooper,In These Arms,spotify:track:2D6ZA92U9vpcVhrRnd8rFN,NaN,...,NaN,NaN,NaN,playbtn,endplay,False,False,False,NaN,False
4,2020-02-19T02:13:49Z,"Android OS 9 API 28 (samsung, SM-A505F)",201896,IN,47.15.173.165,Intentions (feat. Quavo),Justin Bieber,Changes,spotify:track:4umIPjkehX1r7uhmGvXiSV,NaN,...,NaN,NaN,NaN,clickrow,endplay,False,False,False,NaN,False


In [62]:
len(df.columns)

23

In [63]:
# 1. Convert the 'ts' column to datetime objects
# The 'Z' at the end stands for UTC, so pandas handles it cleanly
df['ts'] = pd.to_datetime(df['ts'])
# 2. Add 5 hours and 30 minutes to the timestamp
df['ts'] = df['ts'] + pd.Timedelta(hours=5, minutes=30)

# 3. Format it back to a string with a space instead of 'T'
df['ts'] = df['ts'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Remove 'spotify:track:' from the column (replace it with an empty string '')
df['spotify_track_uri'] = df['spotify_track_uri'].str.replace('spotify:track:', '', regex=False)

In [64]:
# Remove the column
df = df.drop(columns=['conn_country', 
                      'ip_addr',
                      'episode_name',
                      'episode_show_name', 
                      'spotify_episode_uri',
                      'audiobook_title',
                      'audiobook_uri',
                      'audiobook_chapter_uri',
                      'audiobook_chapter_title',
                      'offline_timestamp'])

In [65]:
df.isnull().sum()

ts                                    0
platform                              0
ms_played                             0
master_metadata_track_name           88
master_metadata_album_artist_name    88
master_metadata_album_album_name     88
spotify_track_uri                    88
reason_start                          0
reason_end                            0
shuffle                               0
skipped                               0
offline                               2
incognito_mode                        0
dtype: int64

In [66]:
cols = ['master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'offline']
df[df[cols].isnull().any(axis=1)]
# removing those rows
df = df.dropna(subset=cols)

In [67]:
df.isnull().sum()

ts                                   0
platform                             0
ms_played                            0
master_metadata_track_name           0
master_metadata_album_artist_name    0
master_metadata_album_album_name     0
spotify_track_uri                    0
reason_start                         0
reason_end                           0
shuffle                              0
skipped                              0
offline                              0
incognito_mode                       0
dtype: int64

In [73]:
# Renaming columns

df = df.rename(columns={
    'master_metadata_track_name': 'track_name',
    'master_metadata_album_artist_name': 'artist_name',
    'master_metadata_album_album_name' : 'album_name',
})


In [74]:
# created hours_played for easy analytics
df['hours_played'] = df['ms_played'] / (1000 * 60 * 60)

In [75]:
df.isnull().sum()

ts                   0
platform             0
ms_played            0
track_name           0
artist_name          0
album_name           0
spotify_track_uri    0
reason_start         0
reason_end           0
shuffle              0
skipped              0
offline              0
incognito_mode       0
hours_played         0
dtype: int64

In [78]:
# Exporting To SSMS 
import pandas as pd
from sqlalchemy import create_engine

server = r"servername\SQLEXPRESS"
database = "Spotify_kg"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes"
)

engine = create_engine(connection_string)

In [79]:
with engine.connect() as conn:
    print("Connected successfully!")

Connected successfully!


In [80]:
df.to_sql(
    "spotify_ps_dataset",
    con=engine,
    schema="dbo",
    if_exists="replace",
    index=False)

15